# v2.07 — v2.06 with one horizon: the next-day trend only (validation only)

**One change on top of v2.06** (Siwat, 2026-11-02, `configs/v2_07.json`):
- **What changes:** the model predicts the 1-day trend only (up / flat / down for the next trading day). The trend head's last layer has 3 outputs instead of 9, and the kept epoch and early stopping use the 1-day hold-out macro-F1 alone.
- **What stays:** everything else is exactly v2.06: no price head, news-robust training (p 0.2, whole window), the same rows, seeds and settings.

**Why:** in v2.06 the 1-day score rose significantly (+0.016) when the price task was removed, while the 5-day score fell. v2.07 tests whether the 1-day trend gains more when it no longer shares the model with the 5- and 10-day horizons.

**Ablation:** the signed proposal locks the horizons at 1 / 5 / 10 (scope 4, O1) and requires a price head (L5 / L6). v2.07 departs from both and is recorded as an ablation.

**Runtime:** A100 or L4, Colab only (Linux). Runtime → Run all. About 1 h of training, then about 10 min for the post-hoc step and the comparison.

**Adopt rule** (pre-registered, `claude/log-entry-2026-11-02-v2-07-design.md`), raw models vs v2.06, same rows:
- validation, news on, **1-day horizon**: the paired 95% interval of v2.07 − v2.06 (v2.06: 0.355) is entirely above 0.

Outputs: `Senior/results/v2xx/<GPU>/v2_07*`, `Senior/results/v2_07_posthoc/`, `Senior/results/v2_07_vs_v2_06.json`. Test block sealed.

In [ ]:
!nvidia-smi -L
import torch
name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none'
mem = torch.cuda.get_device_properties(0).total_memory / 2**30 if torch.cuda.is_available() else 0
print(f'GPU: {name}, {mem:.0f} GB')
GPU_TAG = 'A100' if 'A100' in name else 'L4' if 'L4' in name else None
assert GPU_TAG, 'stop: switch the runtime to an A100 or L4 GPU (Runtime -> Change runtime type)'

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import hashlib, json, os, subprocess, sys
from pathlib import Path
SENIOR = '/content/drive/MyDrive/Senior'
PRICE_DIR, RES = f'{SENIOR}/prices', f'{SENIOR}/results'
OUT = f'{RES}/v2xx/{GPU_TAG}'
V207 = f'{OUT}/v2_07.json'
POST = f'{RES}/v2_07_posthoc'                      # v2.07's post-hoc step
BASE = f'{RES}/v2_06_posthoc'                      # v2.06's post-hoc step, the comparison
BASE_RUN = f'{RES}/v2xx/A100/v2_06.json'           # v2.06's run (A100, 2026-11-02)
FWD_INPUTS = f'{RES}/forward_2026/inputs'
for f in [f'{BASE}/step0.json', f'{BASE}/trees.csv.gz', f'{BASE}/parts/val_blank.csv.gz', BASE_RUN, f'{FWD_INPUTS}/forward_prices.csv',
          f'{PRICE_DIR}/set50_union_2019_2025.csv', f'{PRICE_DIR}/yahoo_events_2019_2025.csv']:
    assert os.path.exists(f), f'stop: missing {f}'
os.makedirs(OUT, exist_ok=True)

def stream(args):
    # stream a script's output into the notebook and stop on failure
    p = subprocess.Popen(args, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        if 'defined on' not in line and 'not writable' not in line and 'from_numpy' not in line:
            print(line, end='', flush=True)
    if p.wait() != 0:
        raise SystemExit(f'stop: {" ".join(args[1:3])} failed (exit {p.returncode}); see the lines above')

In [ ]:
!rm -rf /content/Senior-Project && git clone -q https://github.com/Smith8bit/Senior-Project.git /content/Senior-Project
%cd /content/Senior-Project
!git log --oneline -1
for f in ['configs/v2_07.json', 'scripts/train_v2xx.py', 'src/train_utils.py', 'src/run_config.py', 'tests/test_no_price_head.py', 'tests/test_one_horizon.py', 'src/proposal_model.py',
          'scripts/step0_v2_03beta.py', 'scripts/compare_v2xx_posthoc.py', 'scripts/plot_v2xx_training.py']:
    assert os.path.exists(f), f'stop: push the v2.07 code first ({f} missing)'
    print(f, hashlib.sha256(open(f, 'rb').read()).hexdigest()[:12])
!mkdir -p data/raw/prices/siamchart data/raw/prices/yahoo
!cp "$PRICE_DIR/set50_union_2019_2025.csv" data/raw/prices/siamchart/
!cp "$PRICE_DIR/yahoo_events_2019_2025.csv" data/raw/prices/yahoo/
expected = {'data/raw/prices/siamchart/set50_union_2019_2025.csv': '9d7ce4c3e72a974979566dc089d5b8819af54dc24f62b71240f5b4716b556e6f',
            'data/raw/prices/yahoo/yahoo_events_2019_2025.csv': 'd021a58df95fa487f1e49eb50617f6b3187b025b6377235c6180166b1ec84d53',
            'data/derived/sentiment_scores.csv': '7369fd981c4ccffc017d6d5de50ed95c6a4ac4d843bbc6053366029ce614bb1b'}
for path, h in expected.items():
    got = hashlib.sha256(open(path, 'rb').read()).hexdigest()
    print('OK ' if got == h else 'MISMATCH ', path)
    assert got == h, 'stop: a file differs from the frozen copy'
sys.path.insert(0, 'src'); sys.path.insert(0, 'scripts')
import run_config as RC
cfg = RC.load('configs/v2_07.json')
print('v2.07 horizons:', cfg['protocol']['horizons'], '| price head:', cfg['model']['price_head'], '| loss:', json.dumps(cfg['loss']), '| training:', json.dumps(cfg['training']))

In [ ]:
# Tests (about 10 minutes); the notebook stops if any fails
for t in ['tests/test_v2xx.py', 'tests/test_v2xx_training.py', 'tests/test_news_blank.py', 'tests/test_grad_diag.py', 'tests/test_no_price_head.py', 'tests/test_one_horizon.py',
          'tests/test_step0_v2_03beta.py']:
    r = subprocess.run(['python', t], capture_output=True, text=True)
    print(f'--- {t}'); print('\n'.join(l for l in r.stdout.splitlines() if 'defined on' not in l and '    seed' not in l)[-2500:])
    assert r.returncode == 0 and 'FAIL' not in r.stdout, f'stop: {t} failed\n' + r.stderr[-2000:]

In [ ]:
# 1. Train v2.07 (1-day horizon only, no price head): 3 seeds one after another
def done(path, n=3):
    try:
        r = json.load(open(path)); return len(r['runs']) == n and 'summary' in r
    except Exception:
        return False
if done(V207):
    print('v2.07: already trained')
else:
    stream(['python', '-u', 'scripts/train_v2xx.py', '--config', 'configs/v2_07.json', '--out', V207])

In [ ]:
# 2. Summary beside v2.06 and the floors (1-day horizon only), and the learning curves
r = json.load(open(V207)); s, fl = r['summary']['model'], r['floors']
b = json.load(open(BASE_RUN))['summary']['model']
BAR = {'h1': 0.393}
print(f"{'':6}{'v2.07 F1':>18}{'v2.06':>8}{'bar':>7}{'continues':>11}{'reverses':>10}")
for h in ('h1',):
    m = s[h]
    print(f"{h:6}{m['macro_f1']['mean']:>11.4f} ± {m['macro_f1']['std']:.4f}{b[h]['macro_f1']['mean']:>8.4f}{BAR[h]:>7.3f}"
          f"{fl[h]['trend_continues']['macro_f1']:>11.3f}{fl[h]['trend_reverses']['macro_f1']:>10.3f}")
for x in r['runs']:
    print('seed', x['seed'], 'epochs', x['epochs_run'], 'kept', x['best_epoch'], 'hold-out F1', x['best_inner_es_macro_f1'],
          'weights', x['params']['total'], 'minutes', round(x['seconds'] / 60, 1))
from IPython.display import Image, display
import importlib, plot_v2xx_training as PT
for png in (PT.plot(Path(V207)), PT.plot_calls(Path(V207))):
    if png:
        print('saved', png); display(Image(filename=str(png)))

In [ ]:
# 3. Post-hoc step for v2.07 (same script and settings as v2.03beta-v2.06, 1-day horizon only): reproduction check, ts,
#    blend (w on v2.07's own inner hold-out), news blanked and placebo, on validation and 2026
common = ['--run-json', V207, '--fwd-inputs', FWD_INPUTS, '--out-dir', POST, '--label', 'v2.07 post-hoc']
for stage in ('prepare', 'predict', 'trees', 'analyze'):
    stream(['python', 'scripts/step0_v2_03beta.py', stage, *common])

In [ ]:
# 4. v2.07 vs v2.06 on the same rows (1-day horizon), and the pre-registered adopt rule (1-day news-on gain significant)
stream(['python', 'scripts/compare_v2xx_posthoc.py', '--base', BASE, '--new', POST, '--rule', 'h1_up_significant',
        '--out', f'{RES}/v2_07_vs_v2_06.json'])

**Reading the result**
- **ADOPT RULE** (last block): the 1-day news-on gain over v2.06 (0.355) is significant (interval entirely above 0). Raw models.
- Only the 1-day horizon is compared; v2.07 has no 5- or 10-day output and no price head.
- **News blanked**, **ts + blend** and **2026** are reported beside it, as before.
- Send the printout to Claude.